# Accessing and Filtering Metadata with Advanced Spaceborne Thermal Emission and Reflection Radiometer (ASTER) Version 4

## Summary

This tutorial provides a basic outline on how to programmatically access, search, and filter Terra [ASTER](https://terra.nasa.gov/about/terra-instruments/aster) Version 4 metadata. In addition, the tutorial includes an example of how to insert cloud cover values into the metadata records for the ASTER Digital Elevation Model [(AST14DEM)]( https://doi.org/10.5067/ASTER/AST14DEM.004) products. The AST14DEM products are not processed with cloud cover information natively.

ASTER Version 4 metadata follows the NASA Earthdata Unified Metadata Model [(UMM)](https://www.earthdata.nasa.gov/about/esdis/eosdis/cmr/umm). It provides both Collection Level (UMM-C) and Granule Level (UMM-G) metadata in JavaScript Object Notation [(JSON)](https://www.json.org/) format, allowing convenient programmatic access to collection and granule-level metadata using common Python tools.

The UMM metadata files replace the previous .met format metadata files associated with [Version 3](https://www.earthdata.nasa.gov/data/alerts-outages/aster-demand-processing-end-dec.-15-2025) and the Simple Scalable Script-based Science Processor for Missions (S4PM) on-demand architecture.

## Background

Launched on December 18, 1999, aboard NASA's [Terra](https://terra.nasa.gov/) satellite, ASTER has acquired more than 3 million unique images of Earth's surface. ASTER captures images at 15 to 90-meter resolution in 14 different wavelengths, ranging from visible to infrared light.

With the [Final ASTER Processing Campaign](https://www.earthdata.nasa.gov/news/feature-articles/nasa-begins-final-aster-data-processing-campaign), 11 Version 4 ASTER data products are now archived with UMM-C and UMM-G metadata records stored in the Earthdata cloud environment. This enables users to access the ASTER archive using common Python tools and NASA's [`earthaccess`](https://www.earthdata.nasa.gov/data/tools/earthaccess) Python library, which supports authentication, search, and access to NASA Earth science data.

## Learning Objectives

- Search and filter ASTER Version 4 Collection Metadata Records (UMM-C)
- Search and filter ASTER Version 4 Granule Metadata Records (UMM-G)
- Add cloud cover values to AST14DEM Granule Metadata Records and filter them
- Visualize selected metadata records with a subset of granule attributes
- Download granule metadata records for later use

## Data Used  

- ASTER L1A Reconstructed Unprocessed Instrument Data V004 [(AST_L1A)](https://doi.org/10.5067/ASTER/AST_L1A.004)
- ASTER Digital Elevation Model V004 [(AST14DEM)](https://doi.org/10.5067/ASTER/AST14DEM.004)

## Tutorial Outline

1. [**Getting Started**](#getstarted)  
    1.1 Import Packages<br>
    1.2 Setup Current Working Directory  
2. [**Exploring ASTER Version 4 Data Collections using `earthaccess`**](#find_collections)<br>
    2.1 Search for ASTER Version 4 Collections<br>
    2.2 Filtering ASTER Version 4 Collections<br>
    2.3 Search for Data Granules for AST_L1T Data Collection   
3. [**Search for ASTER Version 4 Data Granules using `earthaccess`**](#find_granules)  
    3.1 Define Our Spatial and Temporal Granule Query Parameters<br>
    3.2 Search for Data Granules for AST_L1A Data Collection<br>
    3.3 Examine AST_L1A UMM-G Metadata<br>
4. [**Filtering ASTER Version 4 Metadata**](#filter)   
    4.1 Filter AST_L1A acquisitions by day/night  
    4.2 Filter AST_L1A day acquisitions solar elevation  
5. [**Working with ASTER Digital Elevation Model (AST14DEM)**](#dem)<br>
    5.1 Search for AST14DEM Data Granules<br>
    5.2 Examine AST14DEM UMM-G Metadata<br>
    5.3 Add Cloud Cover to AST14DEM Metadata
    5.4 Filter AST14DEM acquisitions by cloud cover
6. [**Visualizing Results**](#visualize)<br>
    6.1 Extract Product Bounds and Define Color Scheme<br>
    6.2 Make Visualization
7. [**Download UMM Granule Metadata**](#download)<br>
    7.1 Build List of Metadata<br>
    7.2 Download Metadata Records

## 1. Getting Started<a id="getstarted"></a>

### 1.1 Import Packages 

Import the required packages.

In [1]:
import earthaccess
import pandas as pd
import geopandas as gpd
import json
import os
from shapely.geometry import box, Polygon
import folium
from branca.element import Figure
import random

### 1.2 Setup Current Working Directory

In [2]:
working_dir = os.chdir('../../data')

## 2. Exploring ASTER Version 4 Data Collections using `earthaccess` <a id="find_collections"></a>

To find ASTER Version 4 data, we will use the `earthaccess` Python library to search [NASA's Common Metadata Repository (CMR)](https://www.earthdata.nasa.gov/about/esdis/eosdis/cmr) for ASTER Version 4 data collections.

### 2.1 Search for ASTER Version 4 Collections
With `earthaccess`, we can locate all available ASTER Version 4 collections using the keyword argument "ASTER 004". We will use the collection short names for ASTER L1A Reconstructed Unprocessed Instrument Data V004 (AST_L1A) and ASTER Digital Elevation Model V004 (AST14DEM) to locate specific granules later.

The collection search will return all NASA datasets referencing ASTER Version 4, so it is important to understand which collections are of interest and record those short names accordingly.

In [3]:
collections = earthaccess.search_datasets(
    keyword='ASTER',
    version = '004',
    cloud_hosted=True)
print(f'ASTER Version 4 Collections found: {len(collections)}')

ASTER Version 4 Collections found: 13


### 2.2 Filtering ASTER Version 4 Collections
Here, we can filter the collection metadata for attributes we are interested in including, collection short-name, concept-id and version.

In [4]:
collections_info = [
    {
        'short_name': c.summary()['short-name'],
        'collection_concept_id': c.summary()['concept-id'],
        'version': c.summary()['version'],
        'entry_title': c['umm']['EntryTitle']
    }
    for c in collections
]
collections_info

[{'short_name': 'AST_L1T',
  'collection_concept_id': 'C3306888411-LPCLOUD',
  'version': '004',
  'entry_title': 'ASTER Level 1T Precision Terrain Corrected Registered At-Sensor Radiance V004'},
 {'short_name': 'AST_07XT',
  'collection_concept_id': 'C3306884993-LPCLOUD',
  'version': '004',
  'entry_title': 'ASTER L2 Surface Reflectance VNIR and Crosstalk Corrected SWIR V004'},
 {'short_name': 'AST14DEM',
  'collection_concept_id': 'C3306855744-LPCLOUD',
  'version': '004',
  'entry_title': 'ASTER Digital Elevation Model V004'},
 {'short_name': 'AST_07',
  'collection_concept_id': 'C3306877498-LPCLOUD',
  'version': '004',
  'entry_title': 'ASTER L2 Surface Reflectance VNIR and SWIR V004'},
 {'short_name': 'AST_L1B',
  'collection_concept_id': 'C3306888201-LPCLOUD',
  'version': '004',
  'entry_title': 'ASTER L1B Registered Radiance at the Sensor V004'},
 {'short_name': 'AST_05',
  'collection_concept_id': 'C3306858335-LPCLOUD',
  'version': '004',
  'entry_title': 'ASTER L2 Surface 

Finally, we can filter the collection metadata to include only the two products we are interested in: AST_L1A and AST14DEM.

In [5]:
filtered_collections = [c for c in collections_info if 'ASTER L1A' in c.get('entry_title', '') or 'ASTER Digital Elevation Model V004' in c.get('entry_title','')]
filtered_collections

[{'short_name': 'AST14DEM',
  'collection_concept_id': 'C3306855744-LPCLOUD',
  'version': '004',
  'entry_title': 'ASTER Digital Elevation Model V004'},
 {'short_name': 'AST_L1A',
  'collection_concept_id': 'C3306888985-LPCLOUD',
  'version': '004',
  'entry_title': 'ASTER L1A Reconstructed Unprocessed Instrument Data V004'}]

## 3. Search for ASTER Version 4 Data Granules using `earthaccess` <a id="find_ganules"></a>

To find ASTER Version 4 data, we will use the `earthaccess` Python library to search CMR for ASTER Version 4 data granules.

### 3.1 Define Our Spatial and Temporal Granule Query Parameters
Here, we will read our GeoJSON file using `GeoPandas`. We will use the total_bounds property to get the bounding box of our ROI and store it in a Python tuple, which is the expected data type for the bounding_box parameter in the `earthaccess.search_data function`. We will also define our temporal query parameter. These parameters will allow `earthaccess` to return granules that meet our spatial and temporal requirements.

Read in and establish spatial query parameters.

In [6]:
aoi = gpd.read_file('railroad_valley.geojson')
bbox = tuple(list(aoi.total_bounds))
bbox

(np.float64(-115.89358304108943),
 np.float64(38.31559085515974),
 np.float64(-115.33911006153537),
 np.float64(38.5826536407757))

Eastablish temporal query parameters.

In [7]:
temporal = ("2000-05-01T00:00:00", "2008-05-31T23:59:59")

### 3.2 Search for Data Granules for AST_L1A Data Collection
Here, we perform a query to CMR for AST_L1A Version 4 cloud-hosted data that matches our specified spatial and temporal areas of interest using the short name from the L1A collection. We will use our results from `earthaccess` to build a list of UMM-G metadata files.

In [8]:
results = earthaccess.search_data(
    short_name="AST_L1A",
    bounding_box=bbox,
    temporal=temporal,
    provider='LPCLOUD',
    version ='004'
)
print(f'Found {len(results)} Ganules')

Found 151 Ganules


### 3.3 Examine AST_L1A UMM-G Metadata
We can also examine UMM-G metadata, which is returned as a Python dictionary. This is the metadata file that we will carry forward in the tutorial. To achieve this, all the UMM-G metadata dictionaries are added to a list for further filtering.

In [9]:
metadata_records = [g['umm'] for g in results]
metadata_records[0]

{'TemporalExtent': {'SingleDateTime': '2000-05-01T19:00:40.72200Z'},
 'GranuleUR': 'AST_L1A_00405012000190040_20251203144259',
 'AdditionalAttributes': [{'Name': 'ASTERGains',
   'Values': ['01 HGH, 02 HGH, 3N NOR, 3B NOR, 04 NOR, 05 NOR, 06 NOR, 07 NOR, 08 NOR, 09 NOR']},
  {'Name': 'ASTERGRANULEID', 'Values': ['ASTL1A 0005011900401304219029']},
  {'Name': 'ASTERMapProjection', 'Values': ['N/A']},
  {'Name': 'ASTERProcessingCenter', 'Values': ['GDS']},
  {'Name': 'ASTERReceivingCenter', 'Values': ['EDOS']},
  {'Name': 'ASTERSceneOrientationAngle', 'Values': ['10.537433']},
  {'Name': 'ASTERSWIRPointingAngle', 'Values': ['8.503000']},
  {'Name': 'ASTERTIRPointingAngle', 'Values': ['8.556000']},
  {'Name': 'ASTERVNIRPointingAngle', 'Values': ['8.583000']},
  {'Name': 'GenerationDateandTime', 'Values': ['2020-12-18T14:12:26.000Z']},
  {'Name': 'GeometricDBVersion', 'Values': ['03.01']},
  {'Name': 'identifier_product_doi', 'Values': ['10.5067/ASTER/AST_L1A.004']},
  {'Name': 'identifier_

## 4. Filtering ASTER Version 4 Metadata <a id="filter"></a>
Being a Python dictionary, the UMM-G metadata allows filtering based on keys and values within the file. This section will demonstrate how to leverage this by filtering the AST_L1A acquisitions by day/night status as well as by a range of solar elevation values.

### 4.1 Filter AST_L1A acquisitions by day/night
The ASTER instrument acquires data in both day and night modes. In this example, we are returning both day and night acquisitions. Here, we will split our list of UMM-G AST_L1A records into two separate lists: one for day acquisitions and one for night acquisitions. We can accomplish this by using the DayNightFlag key within the DataGranule section of the UMM-G metadata.

In [10]:
day_metadata_records = [record for record in metadata_records if record.get('DataGranule', {}).get('DayNightFlag') == 'Day']
print(f'Found {len(day_metadata_records)} Day Ganules')

Found 117 Day Ganules


In [11]:
night_metadata_records = [record for record in metadata_records if record.get('DataGranule', {}).get('DayNightFlag') == 'Night']
print(f'Found {len(night_metadata_records)} Night Ganules')

Found 34 Night Ganules


### 4.2 Filter AST_L1A day acquisitions solar elevation
We can also examine other values within the UMM-G metadata records and further filter our list of day acquisitions. Here, we will iterate over our list of UMM-G records for day acquisitions and examine the range of values for solar elevation by referencing the SolarElevationAngle parameter in the AdditionalAttributes section of the UMM-G metadata.

In [12]:
solar_angles = []
for record in day_metadata_records:
    for attr in record.get('AdditionalAttributes'):
        if attr.get('Name') == 'SolarElevationAngle':
            solar_angles.append(float(attr['Values'][0]))


min_angle = min(solar_angles)
max_angle = max(solar_angles)
print(f"SolarElevationAngle range: {min_angle}° to {max_angle}°")


SolarElevationAngle range: 26.234694° to 72.8346°


We can then further filter down our list by specifying a minimum and maximum solar elevation. Here we will work with minimum of 30 and a maximum of 70 degrees.

In [13]:
min_angle = 30
max_angle = 70

day_metadata_records_filtered = []

for record in day_metadata_records:
    for attr in record.get('AdditionalAttributes'):
        if attr.get('Name') == 'SolarElevationAngle':
            SolarElevationAngle = float(attr['Values'][0])
    if SolarElevationAngle is not None and min_angle <= SolarElevationAngle <= max_angle:
        day_metadata_records_filtered.append(record)
    
print(f'ASTER Day Granules Records filtered to {len(day_metadata_records_filtered)} Granules SolarElevationAngle range: {min_angle}° to {max_angle}°')


ASTER Day Granules Records filtered to 91 Granules SolarElevationAngle range: 30° to 70°


The result of our filtering of the AST_L1A metadata produces two lists of metadata:

- AST_L1A Day Acquisitions where solar elevation is between 30° and 70°
- AST_L1A Night Acquisitions

## 5. Working with ASTER Digitial Elevation Model (AST14DEM) <a id="dem"></a>
The ASTER Digital Elevation Model (AST14DEM) differs from other ASTER Version 4 products in that the cloud cover percentage of the acquisition is not recorded. However, this missing information can be obtained from the corresponding AST_L1A acquisitions, as the AST14DEM is an input to them.

### 5.1 Search for AST14DEM Data Granules
We perform a query to CMR for the AST14DEM Collection Version 4 to retrieve cloud-hosted data that matches our specified spatial and temporal areas of interest, using the short name from the AST14DEM collection. The results obtained through `earthaccess` will then be used to build a list of UMM-G metadata files.

In [14]:
dem_results = earthaccess.search_data(
    short_name="AST14DEM",
    bounding_box=bbox,
    temporal=temporal,
    provider='LPCLOUD',
    version ='004',
)
print(f'Found {len(dem_results)} DEM Ganules')

Found 135 DEM Ganules


### 5.2 Examine AST14DEM UMM-G Metadata
Here we examine the UMM-G metadata, which is returned as a Python dictionary. Note that there is currently no cloud cover percentage present.

In [15]:
dem_metadata_records = ([g['umm'] for g in dem_results])
dem_metadata_records[0]

{'TemporalExtent': {'SingleDateTime': '2000-05-01T19:00:40.72200Z'},
 'GranuleUR': 'AST14DEM_00405012000190040_20251203144820',
 'AdditionalAttributes': [{'Name': 'ASTERSWIRPointingAngle',
   'Values': ['8.503000']},
  {'Name': 'ASTERTIRPointingAngle', 'Values': ['8.556000']},
  {'Name': 'ASTERVNIRPointingAngle', 'Values': ['8.583000']},
  {'Name': 'BackgroundValue', 'Values': ['-9999 ']},
  {'Name': 'DataType', 'Values': ['16 bit']},
  {'Name': 'DEMType', 'Values': ['Relative']},
  {'Name': 'identifier_product_doi', 'Values': ['10.5067/ASTER/AST14DEM.004']},
  {'Name': 'identifier_product_doi_authority', 'Values': ['https://doi.org']},
  {'Name': 'LocalGranuleID',
   'Values': ['AST14DEM_00405012000190040_20251203144820.tif']},
  {'Name': 'MapProjection', 'Values': ['UTM']},
  {'Name': 'NumberOfLines', 'Values': ['2442']},
  {'Name': 'NumberOfSamples', 'Values': ['2592']},
  {'Name': 'OutputCellSpacing', 'Values': ['30m']},
  {'Name': 'ResamplingMethod', 'Values': ['BL']},
  {'Name': 

### 5.3 Add Cloud Cover to AST14DEM Metadata
In this section, we will add cloud cover percentage values to the UMM-G metadata files for AST14DEM. This enhancement will allow us to filter these records based on an acceptable cloud cover threshold.
To achieve this, we will retrieve the cloud cover values from the AST_L1A products we worked with earlier. Since AST_L1A is the input for AST14DEM, the cloud cover values from AST_L1A are identical to those for AST14DEM. This is possible because all ASTER Version 4 granule IDs carry forward their acquisition timestamp, which we can use as a common key to join the cloud cover data from AST_L1A to AST14DEM.
Example:


AST14DEM granule ID:
- AST14DEM_00405012000190040_20251203144820
- Common key: 00405012000190040

AST_L1A granule ID:
- AST_L1A_00405012000190040_20251203144259
- Common key: 00405012000190040

Here, we create a function that will join the cloud cover values from the AST_L1A products to the AST14DEM metadata.

In [16]:
def merge_cloud_cover(dem_metadata_records, metadata_records):
    """
    Adds CloudCover from AST_L1B dicts into AST14DEM dicts based on matching token in GranuleUR.
    """
    # 1) Build a lookup map from L1B join_key -> CloudCover
    gran_map = {}
    for gran in metadata_records:
        granule = gran.get("GranuleUR", "")
        if granule:
            key = granule.split("_")[2]  # e.g., '00401052001185055'
            gran_map[key] = gran.get("CloudCover")
   
    # 2) Merge and filter
    filtered_list = []
    for dem in dem_metadata_records:
        granule = dem.get("GranuleUR", "")
        if granule:
            key = granule.split("_")[1]
            cloud_cover = gran_map.get(key)
            if cloud_cover is not None:  # Only keep if CloudCover exists
                dem["CloudCover"] = cloud_cover
                filtered_list.append(dem)

    return filtered_list


Here we call the function to join the cloud cover value to the AST14DEM UMM-G.

In [17]:
dem_cloud_cover = merge_cloud_cover(dem_metadata_records, metadata_records)
print(f'Added Cloud Cover to  {len(dem_cloud_cover)} DEM Ganules')

Added Cloud Cover to  117 DEM Ganules


Now when we examine a AST14DEM UMM-G metadata record it now contains cloud cover.

In [18]:
dem_cloud_cover[0]

{'TemporalExtent': {'SingleDateTime': '2000-05-01T19:00:40.72200Z'},
 'GranuleUR': 'AST14DEM_00405012000190040_20251203144820',
 'AdditionalAttributes': [{'Name': 'ASTERSWIRPointingAngle',
   'Values': ['8.503000']},
  {'Name': 'ASTERTIRPointingAngle', 'Values': ['8.556000']},
  {'Name': 'ASTERVNIRPointingAngle', 'Values': ['8.583000']},
  {'Name': 'BackgroundValue', 'Values': ['-9999 ']},
  {'Name': 'DataType', 'Values': ['16 bit']},
  {'Name': 'DEMType', 'Values': ['Relative']},
  {'Name': 'identifier_product_doi', 'Values': ['10.5067/ASTER/AST14DEM.004']},
  {'Name': 'identifier_product_doi_authority', 'Values': ['https://doi.org']},
  {'Name': 'LocalGranuleID',
   'Values': ['AST14DEM_00405012000190040_20251203144820.tif']},
  {'Name': 'MapProjection', 'Values': ['UTM']},
  {'Name': 'NumberOfLines', 'Values': ['2442']},
  {'Name': 'NumberOfSamples', 'Values': ['2592']},
  {'Name': 'OutputCellSpacing', 'Values': ['30m']},
  {'Name': 'ResamplingMethod', 'Values': ['BL']},
  {'Name': 

### 5.4 Filter AST14DEM acquisitions by cloud cover
Now that we have added cloud cover information to the AST14DEM UMM-G metadata files, we can filter them by cloud cover. In this step, we will remove all acquisitions from the AST14DEM list that have a cloud cover greater than zero.

In [19]:
filtered_dems = [dem for dem in dem_cloud_cover if dem.get("CloudCover") == 0]
print(f'Filtered down to  {len(filtered_dems)} DEM Ganules with zero cloud cover')

Filtered down to  29 DEM Ganules with zero cloud cover


## 6. Visualizing Results <a id="visualize"></a>
In this section, we will create a visualization of our three filtered ASTER data products—Day Acquisitions (filtered by sun elevation), Night Acquisitions, and AST14DEM granules with a cloud cover percentage of zero. We will plot the geometry of each product and attach key metadata attributes to each geometry. All information will be extracted from the UMM‑G metadata files.

### 6.1 Extract Product Bounds and Define Color Scheme
First, we will create a function to extract the corner coordinates for each product from the UMM‑G files. Second, we will create a function that assigns a random color to each product for the visualization.

This function extracts the corner point coordinates for each product from the UMM-G file.

In [20]:
def create_polygon(record):
    geometry = (record.get("SpatialExtent", {})
                .get("HorizontalSpatialDomain", {})
                .get("Geometry", {}))
    if "GPolygons" in geometry:
        for gpoly in geometry["GPolygons"]:
            points = [
                (p["Longitude"], p["Latitude"])
                for p in gpoly.get("Boundary", {}).get("Points", [])]
            if points:
                polygon = Polygon(points)
                latlon = [(y, x) for x, y in polygon.exterior.coords]
    return latlon

This function creates assignments a random color to product bounds in the visualization.

In [21]:
def random_hex():
    return "#{:06x}".format(random.randint(0, 0xFFFFFF))

### 6.2 Make Visualization
We now present our visualization. Each product’s spatial bounds is plotted on an interactive map, with key metadata displayed for each product. The visualization includes a layer control menu on the right side, allowing users to toggle individual product layers on or off.
The products shown in the visualization are:

- ASTER L1A Day Acquisitions (filtered by solar elevation)
- ASTER L1A Night Acquisitions
- ASTER Digital Elevation Model (AST14DEM) Products with 0% cloud cover

In [22]:
fig = Figure(width="750px", height="375px")
# Create Map
m = folium.Map(tiles=None)
fig.add_child(m)

# Add Basemap
folium.TileLayer(
    tiles=(
        "https://server.arcgisonline.com/ArcGIS/rest/"
        "services/World_Imagery/MapServer/tile/{z}/{y}/{x}"
    ),
    name="ESRI Satellite",
    attr="Esri",
    overlay=False,
    control=True
).add_to(m)
fg = folium.FeatureGroup(name="ASTER Day")
style_kw = {"fillOpacity": 0.1, "weight": 1}
for i, record in enumerate(day_metadata_records_filtered):
    tooltip_dict =  tooltip_dict = {"Index": i, "Granule ID": record['GranuleUR'], "Date Acquired":record['TemporalExtent']['SingleDateTime'],
                                   "Cloud Cover": record['CloudCover'], "Day Night Flag":record['DataGranule']['DayNightFlag']}
    html = "<br>".join(f"<b>{k}</b>: {v}" for k, v in tooltip_dict.items())
    color = random_hex()
    latlon = create_polygon(record)
    folium.Polygon(latlon, 
                   categorical=True,
                   tooltip = html,
                   style_kwds={**style_kw},
                   color = color,
                   name="ASTER (Day)",).add_to(fg)
    fg.add_to(m)
    m.fit_bounds(bounds=fg.get_bounds())

fg = folium.FeatureGroup(name="ASTER Night")
for i, record in enumerate(night_metadata_records):
    tooltip_dict =  tooltip_dict = {"Index": i, "Granule ID": record['GranuleUR'], "Date Acquired":record['TemporalExtent']['SingleDateTime'],
                                   "Cloud Cover": record['CloudCover'], "Day Night Flag":record['DataGranule']['DayNightFlag']}
    html = "<br>".join(f"<b>{k}</b>: {v}" for k, v in tooltip_dict.items())
    color = random_hex()
    latlon = create_polygon(record)
    folium.Polygon(latlon, 
                   categorical=True,
                   tooltip = html,
                   style_kwds={**style_kw},
                   color = color,
                   name="ASTER (Night)",).add_to(fg)
    fg.add_to(m)
    m.fit_bounds(bounds=fg.get_bounds())

fg = folium.FeatureGroup(name="ASTER Zero Cloud Cover DEMs")
for i, record in enumerate(filtered_dems):
    tooltip_dict =  tooltip_dict = {"Index": i, "Granule ID": record['GranuleUR'], "Date Acquired":record['TemporalExtent']['SingleDateTime'],
                                   "Cloud Cover": record['CloudCover'], "Day Night Flag":record['DataGranule']['DayNightFlag']}
    html = "<br>".join(f"<b>{k}</b>: {v}" for k, v in tooltip_dict.items())
    color = random_hex()
    latlon = create_polygon(record)
    folium.Polygon(latlon, 
                   categorical=True,
                   tooltip = html,
                   style_kwds={**style_kw},
                   color = color,
                   name="ASTER (Filtered DEMSs)",).add_to(fg)
    fg.add_to(m)
    m.fit_bounds(bounds=fg.get_bounds())
folium.LayerControl().add_to(m)
m.add_child(folium.LayerControl(collapsed=False))
fig


## 7. Download UMM Granule Metadata <a id="download"></a>
In this section, we will download the UMM‑G metadata records for our three filtered lists:

- Day Metadata Records (filtered by solar elevation)
- Night Metadata Records
- DEM Metadata Records (filtered by cloud cover)

### 7.1 Build List of Metadata
Here we will build a list of all the UMM-G metadata records we want to download locally.

In [23]:
metadata_downloads = day_metadata_records_filtered.copy()
metadata_downloads.extend(night_metadata_records)
metadata_downloads.extend(filtered_dems)
print(f'{len(metadata_downloads)} Metadata Records to Download')

154 Metadata Records to Download


### 7.2 Download Metadata Records
Here, we iterate over our list of UMM‑G metadata records and download them locally. Each file is saved in JSON format and includes the granule ID.

In [24]:
for dict in metadata_downloads:
    name = dict['GranuleUR']
    with open(f'{name}.json', 'w', encoding='utf-8') as f:
        json.dump(dict, f, indent=4)

Success! You have learned how to search, filter, visualize and download ASTER Version 4 Collection and Granule Metadata. You can now replace the collection short name, GeoJSON file, and temporal range in Section 3 with your own inputs and re-run the notebook.



## Contact Info  

Email: LPDAAC@usgs.gov  
Voice: +1-866-573-3222  
Organization: Land Processes Distributed Active Archive Center (LP DAAC)¹  
Website: <https://www.earthdata.nasa.gov/centers/lp-daac>  

¹Work performed under USGS contract G15PD00467 for NASA contract NNG14HH33I.